# Direct Research — Generación de Knowledge Base

Enfoque simplificado: una sola llamada a Gemini con instrucciones de investigación libre.
Sin pipeline de 4 pasos. Sin validator que filtre experiencias.

In [5]:
import sys
sys.path.append('../')
import json
from src.core.gemini_processor import GeminiProcessor
from src.core.prompt_utils import read_prompt_from_file, replace_variables, validate_prompt_variables
from src.core.base_knowledge import format_technical_specs
from src.sources.inventory.app import Inventory
from src.config.settings import COUNTRY

db_loader = Inventory()
gemini = GeminiProcessor()

## 1. Cargar base de datos

In [6]:
df_data_base = db_loader.load_db_from_country_selected()

CO


## 2. Seleccionar modelo

Cambiar el `code` según el modelo a procesar.

In [7]:
df_modelo = df_data_base[df_data_base["code"] == "CO2961-hero-hunk-125-r"]
df_modelo

,date,code,brand,model,year,title,type,technical_specs,publication_url,publication_image_url
70,09/04/2026,CO2961-hero-hunk-125-r,Hero,Hunk 125 R,2026,Hero Hunk 125R,Urbana,"[{'key': 'ignition', 'value': 'CDI (Ignición p...",https://www.galgo.com/co/motos/CO2961-hero-hun...,https://images.ctfassets.net/8zlbnewncp6f/58pe...


## 3. Extraer variables del modelo

In [8]:
for index, row in df_modelo.iterrows():
    BRAND = row["brand"]
    MODEL = row["model"]
    TITLE = row["title"]
    YEAR = "2025"
    TYPE = row["type"]
    PAIS = {"CO": "Colombia", "MX": "Mexico", "CL": "Chile"}.get(COUNTRY)
    TECHNICAL_SPECS = row["technical_specs"]
    CODE = row["code"]

print(f"Marca: {BRAND}")
print(f"Modelo: {MODEL}")
print(f"Title: {TITLE}")
print(f"Año: {YEAR}")
print(f"País: {PAIS}")
print(f"Tipo: {TYPE}")
print(f"Code: {CODE}")

Marca: Hero
Modelo: Hunk 125 R
Title: Hero Hunk 125R
Año: 2025
País: Colombia
Tipo: Urbana
Code: CO2961-hero-hunk-125-r


## 4. Construir prompt

In [9]:
PROMPT_PATH = "../src/data/input/prompts/direct_research_template.md"

ficha_formateada = format_technical_specs(TECHNICAL_SPECS)

prompt = replace_variables(read_prompt_from_file(PROMPT_PATH), {
    "{MARCA}": BRAND,
    "{MODELO}": MODEL,
    "{AÑO}": str(YEAR),
    "{PAIS}": PAIS,
    "{TIPO}": TYPE,
    "{TITLE}": TITLE,
    "{FICHA TECNICA}": ficha_formateada,
})

# Validar que no queden placeholders sin reemplazar
info = validate_prompt_variables(prompt)
if not info["valid"]:
    raise ValueError(f"Variables sin reemplazar: {info['missing_variables']}")

print(prompt)

# ROL

Eres un investigador especializado en experiencias reales de motocicletas en mercados latinoamericanos.
Tu trabajo es investigar a fondo la Hero Hunk 125 R en Colombia y generar una base de conocimiento
experiencial basada en lo que reportan propietarios reales: cómo se siente, qué falla, qué enamora,
qué decepciona.

Tu enfoque es vivencial y sentimental: cómo se siente VIVIR con la moto, NO describir fichas técnicas.
Menciona conceptos técnicos SOLO cuando expliquen sensaciones, problemas o decisiones de compra.

---

# PROCESO DE INVESTIGACIÓN

Antes de generar el reporte, realiza búsquedas activas en este orden. Usa los resultados de TODAS
las búsquedas para construir el reporte:

1. "Hero Hunk 125R Colombia opiniones usuarios"
2. "Hero Hunk 125R Colombia experiencia propietario"
3. "Hero Hunk 125R Colombia problemas fallas"
4. "Hero Hunk 125R Colombia foro review"
5. "Hero Hunk 125R vs competidores Colombia"
6. "Hero Hunk 125R Colombia precio usado reventa"

Reglas de inves

## 5. Ejecutar investigación

In [10]:
knowledge_base = gemini.send_prompt(prompt)
print(knowledge_base)

c:\Users\JTRUJILLO\Desktop\utiles\Proyectos\deep_research_models\notebooks\..\src\core\gemini_processor.py:56: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.client.interactions.create(


[SEGMENTO]

Coincide con tipo declarado

[SENTIMIENTO]

La Hero Hunk 125 R genera sentimientos mayormente positivos por su diseño imponente que simula mayor cilindraje y su excelente economía de combustible. Sin embargo, existe una frustración inicial frecuente debido a problemas de carburación de fábrica en ciudades de altura, lo que exige un ajuste obligatorio para disfrutarla plenamente.

[SENSACIONES]

Estabilidad: Destaca ampliamente en este aspecto gracias a sus llantas inusualmente anchas para el segmento y su peso, brindando una sensación de gran aplomo en carretera y seguridad en curvas.
Vibraciones: El motor con tecnología EBT mantiene las vibraciones al mínimo en rangos bajos y medios, proporcionando un manejo urbano suave, aunque a altas revoluciones pueden sentirse ligeras vibraciones en los plásticos frontales.
Frenado: Sorprendentemente efectivo y seguro a pesar de tener freno de tambor trasero, gracias al sistema de frenos combinados (IBS) que distribuye la fuerza, y a 

## 6. Guardar output

In [11]:
nombre_archivo = f'../src/data/output/gemini/{COUNTRY}-{BRAND}_{MODEL}-direct.md'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    f.write(knowledge_base)
print(f"Guardado en: {nombre_archivo}")

Guardado en: ../src/data/output/gemini/CO-Hero_Hunk 125 R-direct.md
